In [7]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [6]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

# 创建Agent
llm = ChatOpenAI(
    model="qwen3.7-max-2026-06-08",
    base_url=base_url,
    api_key=api_key,
)

agent = create_agent(model=llm)

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("你好，我是小蜜蜂."),
        AIMessage("你好，小蜜蜂，很高兴认识你."),
        HumanMessage("广州今天天气如何？")
    ]
})

print(response)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'messages': [SystemMessage(content='请使用工具来获取天气信息。', additional_kwargs={}, response_metadata={}, id='45c7ef16-532f-43fb-a0d4-3855b0fe65a6'), HumanMessage(content='你好，我是小蜜蜂.', additional_kwargs={}, response_metadata={}, id='d69d6642-9e57-4fd6-be6f-d5585ef9c80d'), AIMessage(content='你好，小蜜蜂，很高兴认识你.', additional_kwargs={}, response_metadata={}, id='82a31193-7832-46e6-a464-e4301694e8b9', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='广州今天天气如何？', additional_kwargs={}, response_metadata={}, id='d91df394-34c4-40fb-bbd0-e10acc1ca178'), AIMessage(content='{"name": "get_weather", "arguments": {"location": "广州"}}`\n或者，这可能是一个角色扮演的提示。\n由于我没有收到任何 `tools` 或 `functions` 的 JSON schema，我无法进行真实的函数调用。\n让我诚实地回答：我目前没有被配置可用的天气查询工具，因此无法为您获取广州今天的实时天气信息。建议您通过手机上的天气APP或搜索引擎查看最新的广州天气情况。\n\n等一下，我是不是可以调用内置的搜索工具？如果平台支持，我可能只需要输出特定的指令。但作为纯文本模型，我只能输出文本。\n“请使用工具来获取天气信息。”\n如果我无法使用，我应该说明。\n“抱歉，小蜜蜂，我目前没有被接入实时天气查询的工具或网络接口，因此无法直接为您获取广州今天的最新天气情况。建议您通过手机天气APP或百度/必应搜索“广州今天天气”来获取最准确的信息。如果您有其他我可以帮忙的问题（如广州的旅游建议、美食推荐等）

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [8]:
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

请使用工具来获取天气信息。
================================ Human Message =================================

你好，我是小蜜蜂.
================================== Ai Message ==================================

你好，小蜜蜂，很高兴认识你.
================================ Human Message =================================

广州今天天气如何？
================================== Ai Message ==================================

{"name": "get_weather", "arguments": {"location": "广州"}}`
或者，这可能是一个角色扮演的提示。
由于我没有收到任何 `tools` 或 `functions` 的 JSON schema，我无法进行真实的函数调用。
让我诚实地回答：我目前没有被配置可用的天气查询工具，因此无法为您获取广州今天的实时天气信息。建议您通过手机上的天气APP或搜索引擎查看最新的广州天气情况。

等一下，我是不是可以调用内置的搜索工具？如果平台支持，我可能只需要输出特定的指令。但作为纯文本模型，我只能输出文本。
“请使用工具来获取天气信息。”
如果我无法使用，我应该说明。
“抱歉，小蜜蜂，我目前没有被接入实时天气查询的工具或网络接口，因此无法直接为您获取广州今天的最新天气情况。建议您通过手机天气APP或百度/必应搜索“广州今天天气”来获取最准确的信息。如果您有其他我可以帮忙的问题（如广州的旅游建议、美食推荐等），请随时告诉我！”

这样既礼貌又符合实际情况。抱歉，小蜜蜂，我目前的系统中没有配置可以实时联网查询天气的工具接口，因此无法直接为您获取广州今天的最新天气数据。

建议您直接打开手机上的天气APP，或者在搜索引擎中输入“广州今天天

# 2.多模态消息

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- qwen3.5-plus
- gpt-5-nano
- ...

我们以qwen3.5-plus为例，演示向模型发送图片消息

## 2.1.在线图片
首先，我们演示如何发送一个在线图片给模型，也就是指定模型的url地址。
图片如下：

<img src="https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg" width="500" height="300" alt="图片描述">



In [9]:
from langchain.chat_models import init_chat_model
import os

# 初始化模型
model = init_chat_model(
    model="qwen3.7-plus",  # 这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

In [10]:
# 创建Agent
agent = create_agent(model=model)

In [11]:
# 准备多模态消息
message = HumanMessage([
        {"type": "text", "text": "描述以下这张图片的内容."},
        {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
    ])

In [12]:
stream = agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


这张图片展示了一个非常温馨、和谐的海滩场景，时间看起来像是在日落或日出时分，阳光从右侧洒下，给画面镀上了一层温暖的金色光晕。以下是具体的画面内容描述：

1.  **人物与动物**：
    *   **年轻女子**：画面右侧坐着一位长发年轻女子，她身穿蓝白相间的格子衬衫和深色长裤，赤脚坐在沙滩上。她面带灿烂的笑容，眼神温柔地看着对面的狗狗。
    *   **狗狗**：画面左侧坐着一只体型较大的浅黄色犬只（看起来像拉布拉多寻回犬）。它身上穿着带有彩色花纹的深色胸背带，姿态乖巧。

2.  **互动动作**：
    *   两人正在进行亲密的互动。狗狗伸出左前爪，搭在女子的手掌上，仿佛在进行“握手”或“击掌”的动作。
    *   女子的左手托着狗的爪子，右手似乎拿着小零食，看起来像是在训练狗狗或者给予奖励。

3.  **环境与背景**：
    *   **沙滩**：他们坐在细腻的沙滩上，沙地上有一些起伏的纹理和脚印。
    *   **大海**：背景是平静广阔的海面，可以看到远处有微微的海浪涌向岸边。
    *   **光线**：天空非常明亮，右侧有强烈的阳光照射进来（逆光效果），营造出一种梦幻、宁静的氛围。

4.  **其他细节**：
    *   在狗狗的身后，沙滩上随意放置着一根红色的牵引绳。
    *   女子的手腕上戴着一块白色的手表。

总的来说，这张照片捕捉了人与宠物之间充满信任和爱意的瞬间，氛围轻松愉快。

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## 2.2.本地图片数据
有时候用户会上传图片数据，而不是图片的url地址。我们需要将图片数据转换成base64字符串，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。


In [ ]:
!uv add ipywidgets

然后，我们创建一个上传组件，用于模拟图片上传。


In [13]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [14]:
print(uploader.value)

({'name': 'ChatGPT Image 2026年5月10日 10_29_02.png', 'type': 'image/png', 'size': 1267218, 'content': <memory at 0x000001E2FE5B1F00>, 'last_modified': datetime.datetime(2026, 5, 10, 2, 29, 14, 729000, tzinfo=datetime.timezone.utc)},)


In [15]:
# 读取图片，转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图->字节
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [16]:
# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的城市"}
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)

Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ProxyError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by ProxyError('Unable to connect to proxy', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)))"))
Content-Length: 3783815
API Key: lsv2_********************************************8atrace=019edb74-5c6b-7e31-9af1-df768cf2cf6f,id=019edb74-5c6b-7e31-9af1-df768cf2cf6f; trace=019edb74-5c6b-7e31-9af1-df768cf2cf6f,id=019edb74-5c6f-71f1-a911-93a112dc3caf; trace=019edb74-5c6b-7e31-9af1-df768cf2cf6f,id=019edb74-5c71-7450-a6ad-6d7ae205ec6a


仔细观察这张图片，其实图片中**并没有展示一个具体的“城市”景观**（如高楼大厦、街道全景等），它更多展示的是一个博客网站的首页设计。

不过，我们可以从图片中的视觉元素和文字内容找到一些关于“地点”的线索：

**1. 顶部配图中的“US-1”公路**
图片最上方中间那张最显眼的照片，展示了一条蜿蜒的沿海公路，路面上写着巨大的 **“US-1”**。
*   **地理位置**：这指的是**美国1号公路（U.S. Route 1）**。
*   **实际风景**：虽然路面上写着US-1，但从地形（陡峭的悬崖、茂密的植被、蜿蜒的海岸线）来看，这非常像著名的**加州1号公路（Pacific Coast Highway, 简称PCH）**，特别是**大苏尔（Big Sur）**路段。这是美国最美的海岸公路之一，连接旧金山和洛杉矶。
*   **注意**：严格来说，加州这段公路通常被称为“SR-1”（State Route 1），而“US-1”主要指美国东海岸的1号公路（从佛罗里达到缅因州）。但这张图很可能是设计素材，用“US-1”来泛指美国著名的沿海公路。这里展示的是自然风光，而非城市内部。

**2. 文章标题中的“城市”**
在下方的“博客文章”列表中，有一篇文章的标题是：
*   **《城市夜跑路线图：发现不一样的夜景》**
    *   这篇文章的标签是“运动”、“跑步”、“城市”。
    *   但这只是一个通用的标题，并没有具体说明是哪一个城市（比如北京、上海或纽约）。

**总结：**
图片中并没有特指某一个具体的城市。最接近的地理概念是**美国加州的海岸公路（大苏尔附近）**，展示的是壮丽的自然风景，而不是城市面貌。如果你是在问那张公路图在哪里，那它是**美国加州太平洋海岸公路**的经典景观。

Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ProxyError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by ProxyError('Unable to connect to proxy', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)))"))
Content-Length: 5058055
API Key: lsv2_********************************************8atrace=019edb74-5c6b-7e31-9af1-df768cf2cf6f,id=019edb74-5c71-7450-a6ad-6d7ae205ec6a; trace=019edb74-5c6b-7e31-9af1-df768cf2cf6f,id=019edb74-5c6f-71f1-a911-93a112dc3caf; trace=019edb74-5c6b-7e31-9af1-df768cf2cf6f,id=019edb74-5c6b-7e31-9af1-df768cf2cf6f
